In [1]:
import ee
import folium
import geemap
import geopandas as gpd
import json
import os

# Inicialize o Earth Engine
ee.Initialize()

date1 = input("digite a primeira data, as datas precisão ser 'Mês e Dia' exemplo: 01-01")
date2 = input("digite a segunda data")
print(f"período {date1} e {date2}")
def obtem_ano(ano):
    # Carrega o shapefile e filtra para o PARNA Serra da Canastra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um objeto Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Função para selecionar a coleção Landsat com base no ano
    def selecionarColecaoLandsat(ano):
        if ano >= 1984 and ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"  # Landsat 5
        elif ano >= 1999 and ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"  # Landsat 7
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"  # Landsat 8
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Parâmetros de entrada
    data_inicio = f"{ano}-{date1}"
    data_fim = f"{ano}-{date2}"

    # Seleciona a coleção com base no ano
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat)

    # Seleciona as bandas para a visualização True Color e NBR
    if ano <= 2012:  # Landsat 5 e 7
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:  # Landsat 8
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    # Obtém o NBR mínimo (severidade máxima)
    nbrMin = nbr.min()

    # Calcula o centróide do PNSC
    centroid = pnsc_ee.geometry().centroid()

    # Inicializa o mapa centrado no centróide do PNSC
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona camadas ao mapa
    Map.addLayer(nbrMin.updateMask(nbrMin.lte(0)), 
             {"min": -1, "max": 0, "palette": ["orange"]}, 
             "NBR (Valores entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")

    # Retorna o mapa atualizado
    return Map

# Exemplo de uso para um ano específico
mapa = obtem_ano(2022)
mapa


período 07-01 e 10-31


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [13]:
import ee
import geemap
import geopandas as gpd
import json
import os

# Inicialize o Earth Engine
ee.Initialize()

def obtem_ano(ano):
    #Define data de mês e dia de inicio e fim
    date1 = input("Digite a primeira data: ")
    date2 = input("Digite a segunda data: ")
    print(f"Período: {date1} e {date2}")
    
    # Carrega o shapefile do PARNA Serra da Canastra e filtra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um objeto Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Carrega o shapefile de focos de calor
    if ano == 2021:
        foco_de_calor = (r"C:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_qmd_inpe_2021-07-01_2021-10-31_27.shp")
    elif ano == 2022:
        foco_de_calor = (r"C:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_qmd_inpe_2022-07-01_2022-10-31_57.shp")
    focos = gpd.read_file(foco_de_calor)
    focos = focos.loc[focos['Estado'] == 'MINAS GERAIS']
    focos_geojson = focos.to_json()

    # Converte o shapefile de focos de calor para um objeto Earth Engine
    focos_ee = ee.FeatureCollection(json.loads(focos_geojson))

    # Função para selecionar a coleção Landsat com base no ano
    def selecionarColecaoLandsat(ano):
        if ano >= 1984 and ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"  # Landsat 5
        elif ano >= 1999 and ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"  # Landsat 7
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"  # Landsat 8
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Parâmetros de entrada
    data_inicio = f"{ano}-{date1}"
    data_fim = f"{ano}-{date2}"

    # Seleciona a coleção com base no ano
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat)

    # Seleciona as bandas para a visualização True Color e NBR
    if ano <= 2012:  # Landsat 5 e 7
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:  # Landsat 8
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    # Obtém o NBR mínimo (severidade máxima)
    nbrMin = nbr.min()

    # Calcula o centróide do PNSC
    centroid = pnsc_ee.geometry().centroid()

    # Inicializa o mapa centrado no centróide do PNSC
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona camadas ao mapa
    Map.addLayer(nbrMin.updateMask(nbrMin.lte(0)), 
                 {"min": -1, "max": 0, "palette": ["orange"]}, 
                 "NBR (Valores entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")

    # Adiciona o shapefile de focos de calor com simbologia em ponto vermelho
    Map.addLayer(focos_ee, {"color": "red"}, "Focos de Calor")

    # Retorna o mapa atualizado
    return Map

# Exemplo de uso para um ano específico
mapa = obtem_ano(2022)
mapa


Período: 07-01 e 10-31


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [4]:
focos = gpd.read_file(r"C:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_qmd_inpe_2022-07-01_2022-10-31_57.shp")
focos = focos.loc[focos['Estado'] == 'MINAS GERAIS']
focos.head()

,DataHora,Satelite,Pais,Estado,Municipio,Bioma,DiaSemChuv,Precipitac,RiscoFogo,Latitude,Longitude,FRP,geometry
0,2022/07/01 16:35:00,AQUA_M-T,Brasil,MINAS GERAIS,SERRA DO SALITRE,Cerrado,20.0,0.0,0.84,-19.30777,-46.80176,18.8,POINT (-46.80176 -19.30777)
1,2022/07/01 16:35:00,AQUA_M-T,Brasil,MINAS GERAIS,UBERLÂNDIA,Cerrado,13.0,0.0,1.00,-19.24165,-48.49339,17.6,POINT (-48.49339 -19.24165)
2,2022/07/01 16:35:00,AQUA_M-T,Brasil,MINAS GERAIS,CHAPADA GAÚCHA,Cerrado,27.0,0.0,1.00,-15.41597,-45.56118,14.7,POINT (-45.56118 -15.41597)
3,2022/07/01 16:35:00,AQUA_M-T,Brasil,MINAS GERAIS,JANAÚBA,Caatinga,62.0,0.0,1.00,-15.46302,-43.32798,7.6,POINT (-43.32798 -15.46302)
4,2022/07/01 16:35:00,AQUA_M-T,Brasil,MINAS GERAIS,NOVA PORTEIRINHA,Cerrado,44.0,0.0,1.00,-15.75724,-43.25532,6.4,POINT (-43.25532 -15.75724)


In [14]:
# Exemplo de uso para um ano específico
mapa = obtem_ano(2021)
mapa

Período: 07-01 e 10-31


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…